# Lab 2 : Answer from the HR policy

*Week 3 · Utrains LLMOps 8-Week Course*

Run each cell from the top. Read the printed output before you run the next cell.


## What we are achieving in this lab

Lab 1 taught **search**. You cut the HR policy into chunks, turned each chunk into a list of numbers, and found the chunks closest to a question.

That is not yet a chatbot. A chatbot must **write a sentence answer**. That writing step is **generate**. You take the chunks search returned, put them in the prompt, and ask a language model (Claude) to answer **only** from those chunks.

If you skip that rule, Claude can invent a leave or expense rule that is not in the policy. At a company, that answer can reach an employee.

**RAG** again, in full:

- **Retrieval** = search (Lab 1)
- **Generation** = write the answer (this lab)
- **Augmented** = the answer is based on the retrieved chunks, not only on the model's training

Two moments in time:

| When | What runs | How often |
|------|-----------|-----------|
| The HR policy file changes | Load, split, embed, store | Once, then again only when the file changes |
| An employee asks a question | Retrieve, then generate | Every question |

**What you will do, in this order**

1. Build the same search index as Lab 1 on `hr_policy.txt`.
2. Ask the reimbursement question. **Print the retrieved chunks** and read them. If these chunks are wrong, the answer will be wrong, even if the sentences sound confident.
3. Send those chunks plus the question to Claude. Check that the answer matches the chunks.
4. Ask a question the HR policy does not answer (a pet-bereavement policy). Search still returns three chunks, because we asked for three. The prompt must tell Claude: if it is not in the chunks, say you cannot find it.

**Keys.** Same pattern as Week 2: `load_dotenv()` reads `.env` in this folder. You need `OPENAI_API_KEY` (embeddings, Lab 1) and `ANTHROPIC_API_KEY` (Claude, this lab). Setup is in [README.md](./README.md).

**Cost.** A few embedding calls and two short Claude replies. Fractions of a cent.


## What each tool does

You already used the first four in Lab 1. Claude is new.

| Step | Plain meaning | What we use |
|------|----------------|-------------|
| Split | Cut the HR policy into chunks of at most 500 characters, with 50 characters of overlap. | `RecursiveCharacterTextSplitter` |
| Embed | Turn each chunk into a list of 1536 numbers. | OpenAI `text-embedding-3-small` |
| Store | Save those lists in this notebook's memory. Restart the kernel and the store is empty. | `InMemoryVectorStore` |
| Retrieve | For a question, return the 3 closest chunks. | `retriever.invoke` |
| Generate | Write a sentence answer from those chunks only. | Claude `claude-haiku-4-5` |

OpenAI does not write the answer. Claude does not create the lists of numbers. That split is on purpose.


### Step 1. Load the API keys

Same as Week 2 and Lab 1. `load_dotenv()` reads the `.env` file in this folder.

You need two keys:

- `OPENAI_API_KEY` — used when we embed chunks and questions
- `ANTHROPIC_API_KEY` — used when Claude writes the answer

If a later cell fails with an authentication error, paste both keys into `.env` and restart the kernel.


In [ ]:
from dotenv import load_dotenv

load_dotenv()  # reads .env from this folder


### Step 2. Build the search index

This is Lab 1 again: load the HR policy, split it, embed each chunk, store the lists of numbers.

`k=3` means: for every question, return the **three** closest chunks. The search always returns three, even if none of them actually answer the question. You will see that in Step 5.


In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

with open("hr_policy.txt", encoding="utf-8") as f:
    POLICY = f.read()

# Lab 1: 500 characters was the usable size for this file.
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunk_texts = splitter.split_text(POLICY)

# Document = the chunk text plus a small label (chunk number, file name).
docs = []
for i, text in enumerate(chunk_texts):
    docs.append(Document(page_content=text, metadata={"chunk": i, "source": "hr_policy.txt"}))

EMBED_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# from_documents = embed every chunk and keep the lists of numbers in this kernel.
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)
# k=3: every search returns three chunks.
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("file   : hr_policy.txt")
print("chunks :", len(docs))
print("embed  :", EMBED_MODEL)
print("store  : InMemoryVectorStore  (this kernel only)")


### Step 3. Retrieve first — read the chunks before you trust an answer

Ask the same reimbursement question as Lab 1. Print the three chunks. Read them.

If the reimbursement section is not in this list, stop. Generating an answer from the wrong chunks will still produce fluent sentences. Those sentences will be wrong.


In [ ]:
QUESTION = "How do I get reimbursed for a $300 train ticket?"

# invoke = embed the question, then return the 3 closest stored chunks.
hits = retriever.invoke(QUESTION)

print("Question:", QUESTION)
print()
print("Retrieved chunks (read these before the answer)")
print()
for i, hit in enumerate(hits, start=1):
    print("--- hit", i, "  chunk", hit.metadata.get("chunk"), " ---")
    print(hit.page_content)
    print()


You should see the reimbursement section: finance portal, PDF receipt, travel under $500. A $300 ticket needs that last sentence.

### Step 4. Generate — Claude answers only from those chunks

This is the new step.

Claude does **not** see the whole HR policy. It sees only the three chunks from Step 3, plus the question.

The **system** message is the standing rule (Week 2): answer from the context; if the answer is not there, say you cannot find it. The **human** (user) message is the chunks and the question.

`temperature=0` means: pick the most likely next words, not a random creative phrasing. For a policy answer, that is what you want.


In [ ]:
from langchain_anthropic import ChatAnthropic

CHAT_MODEL = "claude-haiku-4-5"
llm = ChatAnthropic(model=CHAT_MODEL, temperature=0)

# Join the three chunk texts into one string to put in the prompt.
parts = []
for hit in hits:
    parts.append(hit.page_content)
context = "\n\n---\n\n".join(parts)

messages = [
    (
        "system",
        "You answer from an HR policy excerpt. "
        "Use only facts that appear in the context. "
        "If the context does not contain the answer, say you cannot find it. "
        "Be brief.",
    ),
    (
        "human",
        "Context:\n" + context + "\n\nQuestion: " + QUESTION,
    ),
]

answer = llm.invoke(messages)
print("chat model:", CHAT_MODEL)
print()
print("--- grounded answer ---")
print(answer.content)


Check two things, in this order:

1. Did search return the reimbursement chunk? (Step 3)
2. Did Claude's answer stay inside those chunks? It should mention the finance portal, a PDF receipt, and that travel under $500 does not need pre-approval.

**Grounded** means: you can point at a sentence in the retrieved text that supports the answer. Claude never saw parental leave or PTO in this prompt, so it should not mention them.


### Step 5. A question the HR policy does not answer

Ask about a pet-bereavement policy. That topic is not in `hr_policy.txt`.

Search still returns three chunks, because `k=3`. Those chunks will be the closest *wrong* topics (often time off or PTO, because the question is about a kind of leave).

Print those chunks. Then generate. The system rule should make Claude say it cannot find the answer, instead of inventing a pet policy.


In [ ]:
missing = "What is the company's pet-bereavement policy?"
missing_hits = retriever.invoke(missing)

print("Question:", missing)
print()
print("Retriever still returned:")
print()
for i, hit in enumerate(missing_hits, start=1):
    print("--- hit", i, "  chunk", hit.metadata.get("chunk"), " ---")
    print(hit.page_content)
    print()

parts = []
for hit in missing_hits:
    parts.append(hit.page_content)
missing_context = "\n\n---\n\n".join(parts)

refusal = llm.invoke(
    [
        (
            "system",
            "You answer using ONLY the provided context. "
            "If the answer is not in the context, say you cannot find it. Do not guess.",
        ),
        (
            "human",
            "Context:\n" + missing_context + "\n\nQuestion: " + missing,
        ),
    ]
)
print("--- model reply ---")
print(refusal.content)


The chunks you printed are not about pets. They are the closest leftover topics in the file.

A grounded prompt should refuse. If Claude invents a pet policy from a PTO chunk, that is a **hallucination**: a fluent answer that is not in the source. Lab 3 shows more ways that happens.

Later, production systems add a score cutoff: if nothing is close enough, **your code** says "I don't know" and does not call the model. We do not add that cutoff today.


### Optional. The same loop on a PDF

Companies often keep policies as PDFs, not `.txt` files. If you drop a short PDF next to this notebook and name it `hr_policy.pdf`, this cell reads the text layer with `pypdf`, then splits, stores, and searches the same way.

If you have no PDF, skip the cell. If the PDF is a scan with no text, the pages will print empty. That is a file problem, not an embeddings problem. Week 4 looks at harder PDFs.


In [ ]:
from pypdf import PdfReader

pdf_path = Path("hr_policy.pdf")
if not pdf_path.exists():
    pdf_path = Path("week03") / "hr_policy.pdf"

if not pdf_path.exists():
    print("No hr_policy.pdf found. Skip this cell, or drop a PDF next to the notebook.")
else:
    pages = []
    reader = PdfReader(str(pdf_path))
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages.append(
            Document(
                page_content=text,
                metadata={"source": str(pdf_path), "page": i + 1},
            )
        )
    pdf_chunks = splitter.split_documents(pages)
    pdf_store = InMemoryVectorStore.from_documents(pdf_chunks, embedding=embeddings)
    pdf_hits = pdf_store.as_retriever(search_kwargs={"k": 3}).invoke(QUESTION)
    print("pages :", len(reader.pages))
    print("chunks:", len(pdf_chunks))
    print("file  :", pdf_path)
    print()
    for hit in pdf_hits:
        preview = hit.page_content.replace("\n", " ")
        if len(preview) > 140:
            preview = preview[:140] + "..."
        print("p." + str(hit.metadata.get("page")), preview)


## What you should be able to explain

You should be able to say these in your own words:

- Search finds chunks. Generate writes the answer. Both are required.
- Always print the retrieved chunks before you trust the answer. Wrong chunks mean a wrong answer, even if the wording sounds sure.
- Claude writes the sentences. OpenAI turns text into lists of numbers. Anthropic does not offer an embedding model.
- Search with `k=3` always returns three chunks. The prompt must say: if the answer is not in those chunks, do not guess.

**Try it as a product.** [`handbook-chat/`](./handbook-chat/) is this same loop in a Streamlit app on the same HR policy. See the Week 3 [README.md](./README.md) to run it.

**Lab 3** uses this loop on a messy set of documents, so you can see when search plus generate is not enough.
